# Introduction to AI: E-commerce Conversion Prediction
This notebook complements the lecture slides in `slides/introduction_to_ai/slides.md`.

## Story
An online shop wants to understand **customer segments** (unsupervised learning) and **predict conversions** (supervised learning).
We will use a **simple synthetic e-commerce dataset** with intuitive features.

**Columns**
- `customer_id`
- `age`, `gender`
- `income_k`, `sessions_last30`, `avg_basket`, `time_on_site_min`
- `utm_source`, `device`, `region`, `ui_variant`
- `converted` (label: 1 = purchase, 0 = no purchase)

**Goal:** predict whether a new customer will convert and discuss which UI variant performs best.

## Exercises
- Exercise 1: Explore the dataset and define features/label.
- Exercise 2: Cluster customers with k-means and interpret segments.
- Exercise 3: Train a classifier to predict `converted` and evaluate it.

Notes:
- Data generation and train/test split are already provided.
- `predict_df` is the unlabeled data for your predictions.
- Work through the notebook from top to bottom.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, ConfusionMatrixDisplay

In [ ]:
# Synthetic e-commerce data (already prepared for you)
rng = np.random.default_rng(42)
n = 1200

df_raw = pd.DataFrame(
    {
        "customer_id": np.arange(1, n + 1),
        "age": rng.integers(18, 70, size=n),
        "gender": rng.choice(["female", "male"], size=n, p=[0.52, 0.48]),
        "income_k": np.round(rng.normal(60, 20, size=n).clip(20, 150), 1),
        "sessions_last30": rng.poisson(3, size=n).clip(0, 20),
        "avg_basket": np.round(rng.normal(80, 40, size=n).clip(10, 300), 1),
        "time_on_site_min": np.round(rng.normal(6, 3, size=n).clip(1, 30), 1),
        "utm_source": rng.choice(
            ["organic", "paid", "social", "email", "referral"],
            size=n,
            p=[0.35, 0.25, 0.2, 0.1, 0.1],
        ),
        "device": rng.choice(["mobile", "desktop", "tablet"], size=n, p=[0.6, 0.3, 0.1]),
        "region": rng.choice(["EU", "US", "APAC"], size=n, p=[0.4, 0.35, 0.25]),
        "ui_variant": rng.choice(["A", "B"], size=n, p=[0.5, 0.5]),
    }
)

# Create conversion probability (simple, interpretable signal)
logit = -3.0
logit += 0.03 * (df_raw["age"] - 35)
logit += 0.04 * (df_raw["income_k"] - 50)
logit += 0.15 * df_raw["sessions_last30"]
logit += 0.02 * (df_raw["avg_basket"] - 60)
logit += 0.10 * (df_raw["time_on_site_min"] - 5)
logit += np.where(df_raw["utm_source"] == "email", 0.8, 0.0)
logit += np.where(df_raw["utm_source"] == "paid", 0.3, 0.0)
logit += np.where(df_raw["utm_source"] == "social", -0.2, 0.0)
logit += np.where(df_raw["device"] == "desktop", 0.2, 0.0)
logit += np.where(df_raw["ui_variant"] == "B", 0.25, 0.0)
logit += rng.normal(0, 0.8, size=n)

prob = 1 / (1 + np.exp(-logit))
df_raw["converted"] = rng.binomial(1, prob)

# Train/validation split (students should not change this)
train_df, valid_df = train_test_split(
    df_raw,
    test_size=0.25,
    random_state=42,
    stratify=df_raw["converted"],
)

# Unlabeled data for predictions (competition)
predict_df = valid_df.drop(columns=["converted"]).copy()
y_valid = valid_df["converted"].copy()


def score_predictions(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    print(f"Accuracy: {acc:.3f}")
    return acc


def score_on_validation(y_pred):
    return score_predictions(y_valid, y_pred)


df_raw.head()

In [ ]:
# Exercise 1: Explore and prepare the data

df = df_raw.copy()

display(df.head())
display(df.isna().sum())
print(df.shape)
print("Conversion rate:", round(df["converted"].mean(), 3))

numeric_cols = ["age", "income_k", "sessions_last30", "avg_basket", "time_on_site_min"]
categorical_cols = ["gender", "utm_source", "device", "region", "ui_variant"]
feature_cols = numeric_cols + categorical_cols
cluster_features = numeric_cols

X_train = train_df[feature_cols].copy()
y_train = train_df["converted"].copy()
X_valid = valid_df[feature_cols].copy()
y_valid = valid_df["converted"].copy()

display(X_train.describe())
y_train.value_counts()

## Part A: Unsupervised learning - k-means clustering
We ignore the label and group customers purely by behavior. This mimics a real **segmentation** use case.

### Tasks
1. Scale the features.
2. Fit k-means with a chosen k (start with k=4).
3. Inspect cluster sizes and interpret clusters using feature averages.

In [ ]:
# Exercise 2: K-means clustering

X_cluster = df[cluster_features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

k = 4
kmeans = KMeans(n_clusters=k, n_init=20, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

df_clustered = df.copy()
df_clustered["cluster"] = clusters

display(df_clustered["cluster"].value_counts().sort_index())
display(df_clustered.groupby("cluster")[cluster_features].mean())

In [ ]:
# Visualize clusters with two simple features (no PCA)

plt.figure(figsize=(7, 5))
scatter = plt.scatter(
    df_clustered["income_k"],
    df_clustered["avg_basket"],
    c=df_clustered["cluster"],
    cmap="viridis",
    alpha=0.8,
)
plt.xlabel("Income (k)")
plt.ylabel("Average basket")
plt.title("K-means clusters (income vs basket)")
plt.colorbar(scatter, label="cluster")
plt.show()

## Part B: Supervised learning - classification
Now we use the label `converted` and train a model to **predict** whether a customer converts.
Train/test sets are already defined for you.

### Tasks
1. One-hot encode categorical features.
2. Build a pipeline with scaling + a classifier.
3. Train, predict, and evaluate.

In [ ]:
# Exercise 3: Classification

X_train_enc = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_valid_enc = pd.get_dummies(X_valid, columns=categorical_cols, drop_first=True)

X_train_enc, X_valid_enc = X_train_enc.align(
    X_valid_enc, join="left", axis=1, fill_value=0
)

clf = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=300)),
])

clf.fit(X_train_enc, y_train)
y_pred = clf.predict(X_valid_enc)

score_on_validation(y_pred)
print(classification_report(y_valid, y_pred))

In [ ]:
# Plot the confusion matrix for your classifier

ConfusionMatrixDisplay.from_estimator(clf, X_valid_enc, y_valid, cmap="Blues")
plt.title("Confusion matrix")
plt.show()

## Discussion and extensions
- Try different values of k and justify your choice.
- Compare clusters to `converted` and discuss mismatches.
- Swap the classifier (DecisionTree, RandomForest, SVM) and compare metrics.
- Compare performance with and without feature scaling.
- Which UI variant performs better overall? Does it depend on device or source?